# DTW 유사도 계산 (Google Colab GPU 버전)

약 34,000개 규모의 음성 데이터에 대해 감정(상황) 그룹별로 DTW(Dynamic Time Warping) 거리를
GPU에서 배치(batch) 단위로 병렬 계산합니다.

로컬의 `similarity.py`/`similarity_vectorized.py`와 동일하게 4차년도/5차년도 두 데이터셋을
통합해서 7가지 감정(`happiness, angry, disgust, fear, neutral, sadness, surprise`) 전부를 처리합니다.

**사용 전 준비**
1. Google Drive에 4차년도/5차년도 오디오 폴더와 메타데이터 csv(로컬의 `4차년도.csv`, `5차년도_2차.csv`와
   동일한 컬럼 구조: `wav_id`, `상황` 포함)를 각각 업로드
2. 아래 `설정` 셀에서 `DRIVE_ROOT`를 실제 경로로 수정하고, `DATASETS` 리스트의 각 항목(`audio_dir`,
   `sample_csv`)이 실제 Drive 경로를 가리키도록 수정
3. 런타임 유형을 GPU로 설정 (런타임 > 런타임 유형 변경 > GPU)

**설계 개요**
- Cross-Correlation / Cosine / MSE는 로컬(`similarity_vectorized.py`)에서 행렬 연산으로 이미 빠르게 처리 가능하므로 여기서는 다루지 않음
- DTW는 셀 하나(`D[i,j]`)가 이전 대각선(anti-diagonal)의 값에만 의존하므로, 같은 대각선 위의 모든 셀 + 여러 쌍(pair)을 배치로 묶어 GPU에서 동시에 계산
- 데이터셋마다 상황 라벨 표기가 달라서(4차년도 `anger`/`sad` vs 5차년도 `angry`/`sadness`) `SITUATION_ALIASES`로 하나로 맞춘 뒤 병합
- Colab 세션은 일정 시간 후 끊길 수 있으므로, 처리한 쌍 수를 체크포인트로 저장해서 다시 실행하면 이어서 진행하도록 구성

**튜닝 포인트**
- `RESAMPLE_LENGTH`: DTW는 O(L^2)로 늘어나므로 크게 잡으면(예: 1000) 매우 느려짐. 200 내외를 권장하며, 더 빠르게 하려면 100까지 낮춰도 됨
- `BATCH_SIZE`: 한 번에 GPU에 올릴 쌍(pair)의 수. GPU 메모리가 허용하는 한 크게 잡을수록 전체 처리 시간이 줄어듦 (메모리 사용량은 대략 `BATCH_SIZE * RESAMPLE_LENGTH^2 * 4bytes`)
- 전체 쌍의 수는 그룹 크기의 제곱에 비례해서 늘어나므로(`n개 파일 -> n*(n-1)/2 쌍`), 먼저 작은 서브셋으로 예상 소요 시간을 가늠해보고 전체 실행 여부를 판단할 것을 권장

## 1. Google Drive 마운트 및 GPU 확인

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 2. 설정 (실제 경로/파라미터로 수정)

In [ ]:
import os

# TODO: 실제 Drive 경로로 수정
DRIVE_ROOT = '/content/drive/MyDrive/emotion_voice_dataset'

# 4차년도/5차년도 데이터셋을 통합해서 처리한다. 각 데이터셋은 오디오 폴더와
# 매칭되는 CSV(wav_id -> 상황)를 갖는다. 로컬 similarity.py의 DATASETS와 동일한 구조.
DATASETS = [
    {
        'audio_dir': os.path.join(DRIVE_ROOT, '4차년도'),
        'sample_csv': os.path.join(DRIVE_ROOT, '4차년도.csv'),
        'encoding': 'cp949',
    },
    {
        'audio_dir': os.path.join(DRIVE_ROOT, '5차년도_2차'),
        'sample_csv': os.path.join(DRIVE_ROOT, '5차년도_2차.csv'),
        'encoding': 'cp949',
    },
]

# 데이터셋마다 상황 라벨 표기가 달라서(예: 4차년도 "anger"/"sad" vs
# 5차년도 "angry"/"sadness") 하나로 맞춰준다.
SITUATION_ALIASES = {
    'anger': 'angry',
    'sad': 'sadness',
}

OUTPUT_DIR = os.path.join(DRIVE_ROOT, 'output', 'similarity', 'dtw')
CHECKPOINT_DIR = os.path.join(DRIVE_ROOT, 'output', 'similarity', 'dtw_checkpoint')

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

RESAMPLE_LENGTH = 200   # DTW 전용 리샘플링 길이 (짧을수록 빠름)
BATCH_SIZE = 2000       # 한 배치에서 동시에 처리할 쌍(pair)의 수
DTYPE = torch.float32

PROGRESS_LOG_EVERY = 10  # 몇 배치마다 진행 상황을 출력할지

## 3. 데이터 로드 및 전처리

In [ ]:
import csv
import numpy as np
from scipy.io import wavfile
from scipy.signal import resample

# load_groups()가 채우는, wav_id -> 해당 오디오가 들어있는 데이터셋의 audio_dir.
_WAV_ID_TO_AUDIO_DIR: dict = {}


def load_groups() -> dict:
    groups: dict = {}
    for dataset in DATASETS:
        with open(dataset['sample_csv'], encoding=dataset['encoding'], newline='') as f:
            reader = csv.DictReader(f)
            for row in reader:
                situation = SITUATION_ALIASES.get(row['상황'].strip(), row['상황'].strip())
                wav_id = row['wav_id'].strip()
                groups.setdefault(situation, []).append(wav_id)
                _WAV_ID_TO_AUDIO_DIR[wav_id] = dataset['audio_dir']
    return groups


def get_audio_path(wav_id: str) -> str:
    audio_dir = _WAV_ID_TO_AUDIO_DIR.get(wav_id)
    if audio_dir is None:
        raise KeyError(f"'{wav_id}'의 오디오 폴더를 알 수 없습니다. load_groups()를 먼저 호출했는지 확인하세요.")
    return os.path.join(audio_dir, f'{wav_id}.wav')


def load_waveform(wav_id: str) -> np.ndarray:
    wav_path = get_audio_path(wav_id)
    _, data = wavfile.read(wav_path)
    if data.ndim > 1:
        data = data.mean(axis=1)
    return data.astype(np.float64)


def preprocess(data: np.ndarray, target_len: int = RESAMPLE_LENGTH) -> np.ndarray:
    resampled = resample(data, target_len)
    std = resampled.std()
    if std == 0:
        return resampled - resampled.mean()
    return (resampled - resampled.mean()) / std


def build_group_tensor(wav_ids: list) -> torch.Tensor:
    vectors = [preprocess(load_waveform(wav_id)) for wav_id in wav_ids]
    array = np.vstack(vectors).astype(np.float32)
    return torch.from_numpy(array).to(DEVICE, dtype=DTYPE)

## 4. 배치 anti-diagonal DTW (GPU)

In [ ]:
def batched_dtw(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """x, y: (B, L) 텐서. 반환값: (B,) DTW 거리(경로 길이로 정규화).

    D[i, j]는 대각선 d = i + j 위의 값들끼리는 서로 독립이라 같은 대각선의
    모든 (i, j)를 배치와 함께 한 번에 계산한다 (anti-diagonal wavefront).
    """
    batch_size, length = x.shape
    cost = (x.unsqueeze(2) - y.unsqueeze(1)).abs()  # (B, L, L)

    d_table = torch.full(
        (batch_size, length + 1, length + 1), float('inf'), device=x.device, dtype=x.dtype
    )
    d_table[:, 0, 0] = 0.0

    for d in range(2, 2 * length + 1):
        i_min = max(1, d - length)
        i_max = min(length, d - 1)
        i_idx = torch.arange(i_min, i_max + 1, device=x.device)
        j_idx = d - i_idx

        c = cost[:, i_idx - 1, j_idx - 1]
        up = d_table[:, i_idx - 1, j_idx]
        left = d_table[:, i_idx, j_idx - 1]
        diag = d_table[:, i_idx - 1, j_idx - 1]

        d_table[:, i_idx, j_idx] = c + torch.minimum(torch.minimum(up, left), diag)

    dtw_final = d_table[:, length, length]
    return dtw_final / (2 * length)

## 5. 쌍(pair) 인덱싱 — O(1) 스킵을 위한 조합 공식

`itertools.combinations`로 큰 쌍 목록을 순회하며 스킵하면 재시작할 때마다 오래 걸리므로,
선형 인덱스 `k`에서 바로 `(i, j)`를 계산하는 공식을 사용해 체크포인트에서 즉시 이어갈 수 있게 한다.

In [ ]:
def make_row_starts(n: int) -> np.ndarray:
    row_counts = (n - 1) - np.arange(n - 1)  # i번째 행(첫 원소가 i)에 속하는 쌍의 개수
    row_starts = np.concatenate([[0], np.cumsum(row_counts)])
    return row_starts  # length n, row_starts[-1] == 전체 쌍 개수


def pairs_from_indices(row_starts: np.ndarray, k: np.ndarray) -> tuple:
    i = np.searchsorted(row_starts, k, side='right') - 1
    j = i + 1 + (k - row_starts[i])
    return i, j

## 6. 체크포인트 유틸

In [ ]:
import json


def checkpoint_path(situation: str) -> str:
    return os.path.join(CHECKPOINT_DIR, f'{situation}.json')


def load_checkpoint(situation: str) -> int:
    path = checkpoint_path(situation)
    if os.path.exists(path):
        with open(path, encoding='utf-8') as f:
            return json.load(f)['pairs_done']
    return 0


def save_checkpoint(situation: str, pairs_done: int) -> None:
    with open(checkpoint_path(situation), 'w', encoding='utf-8') as f:
        json.dump({'pairs_done': pairs_done}, f)

## 7. 그룹별 처리 메인 루프

In [ ]:
import time


def process_situation(situation: str, wav_ids: list) -> None:
    n = len(wav_ids)
    if n < 2:
        print(f"경고: '{situation}' 상황 파일이 2개 미만이라 건너뜀.")
        return

    row_starts = make_row_starts(n)
    total_pairs = int(row_starts[-1])

    pairs_done = load_checkpoint(situation)
    if pairs_done >= total_pairs:
        print(f"'{situation}' 이미 완료됨 ({total_pairs}쌍). 건너뜀.")
        return

    print(f"'{situation}': 파일 {n}개, 전체 {total_pairs}쌍, {pairs_done}쌍부터 재개")

    vectors = build_group_tensor(wav_ids)  # (n, L), GPU 상주

    output_path = os.path.join(OUTPUT_DIR, f'{situation}.csv')
    write_header = not os.path.exists(output_path) or pairs_done == 0
    mode = 'w' if write_header else 'a'

    with open(output_path, mode, newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f)
        if write_header:
            writer.writerow(['file_1', 'file_2', 'dtw_distance'])

        batch_idx = 0
        start = pairs_done
        t0 = time.perf_counter()

        while start < total_pairs:
            end = min(start + BATCH_SIZE, total_pairs)
            k = np.arange(start, end)
            i_idx, j_idx = pairs_from_indices(row_starts, k)

            x_batch = vectors[i_idx]
            y_batch = vectors[j_idx]
            distances = batched_dtw(x_batch, y_batch).detach().cpu().numpy()

            for local_i, local_j, dist in zip(i_idx, j_idx, distances):
                writer.writerow([wav_ids[local_i], wav_ids[local_j], float(dist)])

            start = end
            save_checkpoint(situation, start)

            batch_idx += 1
            if batch_idx % PROGRESS_LOG_EVERY == 0 or start >= total_pairs:
                elapsed = time.perf_counter() - t0
                rate = (start - pairs_done) / elapsed if elapsed > 0 else 0
                remaining = (total_pairs - start) / rate if rate > 0 else float('inf')
                print(
                    f"  [{situation}] {start}/{total_pairs}쌍 "
                    f"({rate:.1f}쌍/초, 예상 남은 시간 {remaining/60:.1f}분)"
                )

    print(f"저장 완료: {situation}.csv")


def main() -> None:
    groups = load_groups()
    for situation, wav_ids in groups.items():
        process_situation(situation, wav_ids)


main()

## 참고

- 세션이 끊기면 노트북을 다시 열어 처음부터 순서대로 셀을 재실행하면 된다. `load_checkpoint`가
  `output/similarity/dtw_checkpoint/{situation}.json`에 저장된 진행 상황을 읽어 자동으로 이어서 진행한다.
- 전체 실행 전에 `SAMPLE_CSV`를 파일 수가 적은 서브셋(예: 그룹당 50개)으로 먼저 테스트해서
  `쌍/초` 처리율을 확인하고, 그 값으로 전체 34,000개에 대한 예상 소요 시간을 가늠할 것을 권장한다.
- `BATCH_SIZE`를 늘리면 처리율이 올라가지만 GPU 메모리 부족(OOM)이 나면 절반으로 줄여서 재시도한다.